# HDB Resale Price Regression — Notebook 10: Matched Pairs

Find real transaction pairs that are nearly identical except for one variable, to illustrate the regression findings with concrete examples. Inspired by [Leon Yin's Markup story](https://themarkup.org/still-loading/2022/10/19/dollars-to-megabits-you-may-be-paying-400-times-as-much-as-your-neighbor-for-internet-service) on internet pricing.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/hdb_analysis.csv', low_memory=False)
df['month'] = pd.to_datetime(df['month'])
print(f'Loaded {len(df):,} transactions')

Loaded 50,718 transactions


## Pair 1: Near vs far from columbarium

Find two similar flats — same town, flat type, similar floor and lease — but one is close to a columbarium and one is far. Model predicts +$8 per metre of distance.

In [2]:
# Find pairs: same town, same flat_type, similar storey and lease, different columbarium distance
def find_columbarium_pairs(df, n=5):
    near = df[df['columbarium_dist_m'] < 300].copy()
    far = df[df['columbarium_dist_m'] > 1500].copy()
    
    pairs = []
    for _, flat_near in near.iterrows():
        # Find matching far flat: same town, same type, similar floor and lease
        matches = far[
            (far['town'] == flat_near['town']) &
            (far['flat_type'] == flat_near['flat_type']) &
            (abs(far['storey_mid'] - flat_near['storey_mid']) <= 3) &
            (abs(far['remaining_lease_years'] - flat_near['remaining_lease_years']) <= 5) &
            (abs(far['floor_area_sqm'] - flat_near['floor_area_sqm']) <= 5)
        ]
        if len(matches) > 0:
            best = matches.iloc[(matches['floor_area_sqm'] - flat_near['floor_area_sqm']).abs().argmin()]
            pairs.append({
                'town': flat_near['town'],
                'flat_type': flat_near['flat_type'],
                'near_block': flat_near['block'],
                'near_street': flat_near['street_name'],
                'near_storey': flat_near['storey_range'],
                'near_sqm': flat_near['floor_area_sqm'],
                'near_lease': flat_near['remaining_lease_years'],
                'near_columb_m': round(flat_near['columbarium_dist_m']),
                'near_price': flat_near['resale_price'],
                'far_block': best['block'],
                'far_street': best['street_name'],
                'far_storey': best['storey_range'],
                'far_sqm': best['floor_area_sqm'],
                'far_lease': best['remaining_lease_years'],
                'far_columb_m': round(best['columbarium_dist_m']),
                'far_price': best['resale_price'],
                'price_diff': best['resale_price'] - flat_near['resale_price'],
                'dist_diff_m': round(best['columbarium_dist_m'] - flat_near['columbarium_dist_m']),
            })
        if len(pairs) >= n:
            break
    
    result = pd.DataFrame(pairs)
    return result

columb_pairs = find_columbarium_pairs(df, n=10)
print(f'Found {len(columb_pairs)} pairs\n')

for i, row in columb_pairs.iterrows():
    print(f'--- Pair {i+1}: {row["town"]}, {row["flat_type"]} ---')
    print(f'  NEAR columbarium ({row["near_columb_m"]}m): Blk {row["near_block"]} {row["near_street"]}, '
          f'{row["near_storey"]}, {row["near_sqm"]}sqm, {row["near_lease"]}yr lease -> ${row["near_price"]:,.0f}')
    print(f'  FAR  from columb ({row["far_columb_m"]}m):  Blk {row["far_block"]} {row["far_street"]}, '
          f'{row["far_storey"]}, {row["far_sqm"]}sqm, {row["far_lease"]}yr lease -> ${row["far_price"]:,.0f}')
    print(f'  Price difference: ${row["price_diff"]:+,.0f}  |  Distance gap: {row["dist_diff_m"]:,}m')
    predicted = row['dist_diff_m'] * 8  # model coefficient
    print(f'  Model predicts: ~${predicted:+,.0f} for that distance gap')
    print()

Found 10 pairs

--- Pair 1: BEDOK, 5 ROOM ---
  NEAR columbarium (290m): Blk 114 LENGKONG TIGA, 07 TO 09, 120.0sqm, 64yr lease -> $920,000
  FAR  from columb (1523m):  Blk 648 JLN TENAGA, 07 TO 09, 121.0sqm, 67yr lease -> $888,000
  Price difference: $-32,000  |  Distance gap: 1,233m
  Model predicts: ~$+9,864 for that distance gap

--- Pair 2: BEDOK, 5 ROOM ---
  NEAR columbarium (290m): Blk 114 LENGKONG TIGA, 04 TO 06, 120.0sqm, 64yr lease -> $820,000
  FAR  from columb (1523m):  Blk 648 JLN TENAGA, 07 TO 09, 121.0sqm, 67yr lease -> $888,000
  Price difference: $+68,000  |  Distance gap: 1,233m
  Model predicts: ~$+9,864 for that distance gap

--- Pair 3: BEDOK, 5 ROOM ---
  NEAR columbarium (291m): Blk 116 LENGKONG TIGA, 13 TO 15, 120.0sqm, 64yr lease -> $975,000
  FAR  from columb (3595m):  Blk 164 BEDOK STH RD, 10 TO 12, 122.0sqm, 60yr lease -> $780,000
  Price difference: $-195,000  |  Distance gap: 3,304m
  Model predicts: ~$+26,432 for that distance gap

--- Pair 4: BEDOK, 5 RO

## Pair 2: Block with 4 vs block without 4

Find two similar flats in the same estate — one in a block whose number contains 4, one without. Model predicts a $10,133 discount for block-4.

In [3]:
def find_block4_pairs(df, n=10):
    df['block_str'] = df['block'].astype(str)
    has4 = df[df['block_has_4'] == 1].copy()
    no4 = df[df['block_has_4'] == 0].copy()
    
    pairs = []
    for _, flat4 in has4.iterrows():
        matches = no4[
            (no4['street_name'] == flat4['street_name']) &
            (no4['flat_type'] == flat4['flat_type']) &
            (abs(no4['storey_mid'] - flat4['storey_mid']) <= 3) &
            (abs(no4['remaining_lease_years'] - flat4['remaining_lease_years']) <= 3) &
            (abs(no4['floor_area_sqm'] - flat4['floor_area_sqm']) <= 3)
        ]
        if len(matches) > 0:
            best = matches.iloc[(matches['floor_area_sqm'] - flat4['floor_area_sqm']).abs().argmin()]
            pairs.append({
                'street': flat4['street_name'],
                'flat_type': flat4['flat_type'],
                'block4': flat4['block_str'],
                'block4_storey': flat4['storey_range'],
                'block4_sqm': flat4['floor_area_sqm'],
                'block4_lease': flat4['remaining_lease_years'],
                'block4_price': flat4['resale_price'],
                'other_block': best['block_str'],
                'other_storey': best['storey_range'],
                'other_sqm': best['floor_area_sqm'],
                'other_lease': best['remaining_lease_years'],
                'other_price': best['resale_price'],
                'price_diff': flat4['resale_price'] - best['resale_price'],
            })
        if len(pairs) >= n:
            break
    
    return pd.DataFrame(pairs)

block4_pairs = find_block4_pairs(df, n=10)
print(f'Found {len(block4_pairs)} pairs\n')

for i, row in block4_pairs.iterrows():
    print(f'--- Pair {i+1}: {row["street"]}, {row["flat_type"]} ---')
    print(f'  Block {row["block4"]} (has 4): {row["block4_storey"]}, '
          f'{row["block4_sqm"]}sqm, {row["block4_lease"]}yr lease -> ${row["block4_price"]:,.0f}')
    print(f'  Block {row["other_block"]} (no 4):  {row["other_storey"]}, '
          f'{row["other_sqm"]}sqm, {row["other_lease"]}yr lease -> ${row["other_price"]:,.0f}')
    print(f'  Block-4 discount: ${row["price_diff"]:+,.0f}  |  Model predicts: -$10,133')
    print()

Found 10 pairs

--- Pair 1: ANG MO KIO AVE 10, 3 ROOM ---
  Block 405 (has 4): 10 TO 12, 67.0sqm, 54yr lease -> $350,000
  Block 575 (no 4):  13 TO 15, 67.0sqm, 55yr lease -> $385,000
  Block-4 discount: $-35,000  |  Model predicts: -$10,133

--- Pair 2: ANG MO KIO AVE 10, 3 ROOM ---
  Block 463 (has 4): 01 TO 03, 68.0sqm, 55yr lease -> $350,000
  Block 557 (no 4):  04 TO 06, 68.0sqm, 55yr lease -> $362,000
  Block-4 discount: $-12,000  |  Model predicts: -$10,133

--- Pair 3: ANG MO KIO AVE 10, 3 ROOM ---
  Block 542 (has 4): 01 TO 03, 68.0sqm, 56yr lease -> $352,000
  Block 557 (no 4):  04 TO 06, 68.0sqm, 55yr lease -> $362,000
  Block-4 discount: $-10,000  |  Model predicts: -$10,133

--- Pair 4: ANG MO KIO AVE 10, 3 ROOM ---
  Block 446 (has 4): 04 TO 06, 68.0sqm, 54yr lease -> $355,000
  Block 550 (no 4):  07 TO 09, 68.0sqm, 56yr lease -> $362,888
  Block-4 discount: $-7,888  |  Model predicts: -$10,133

--- Pair 5: ANG MO KIO AVE 10, 3 ROOM ---
  Block 540 (has 4): 04 TO 06, 68.0

## Pair 3: Lease decay in Bukit Timah vs Yishun

Find flats with similar remaining lease in both towns to show how the same lease length commands very different prices — and how much more each year matters in a prime town.

In [4]:
# Compare flats at different lease levels in Bukit Timah vs Yishun
def compare_lease_by_town(df, town1, town2, flat_type='4 ROOM'):
    t1 = df[(df['town'] == town1) & (df['flat_type'] == flat_type)].copy()
    t2 = df[(df['town'] == town2) & (df['flat_type'] == flat_type)].copy()
    
    # Group into lease bands
    bands = [(85, 99, '85-99yr'), (70, 84, '70-84yr'), (55, 69, '55-69yr'), (40, 54, '40-54yr')]
    
    print(f'Comparing {flat_type} flats: {town1} vs {town2}\n')
    print(f'{"Lease band":<12} {town1:>20} {"N":>5} {town2:>20} {"N":>5} {"Gap":>15}')
    print('-' * 82)
    
    prev_gap = None
    for lo, hi, label in bands:
        p1 = t1[(t1['remaining_lease_years'] >= lo) & (t1['remaining_lease_years'] <= hi)]
        p2 = t2[(t2['remaining_lease_years'] >= lo) & (t2['remaining_lease_years'] <= hi)]
        
        if len(p1) > 0 and len(p2) > 0:
            m1 = p1['resale_price'].median()
            m2 = p2['resale_price'].median()
            gap = m1 - m2
            
            change = ''
            if prev_gap is not None:
                change = f'  (gap changed ${gap - prev_gap:+,.0f})'
            
            print(f'{label:<12} ${m1:>18,.0f} {len(p1):>5} ${m2:>18,.0f} {len(p2):>5} ${gap:>13,.0f}{change}')
            prev_gap = gap
        else:
            missing = town1 if len(p1) == 0 else town2
            print(f'{label:<12} (no {flat_type} flats in {missing} at this lease range)')

compare_lease_by_town(df, 'BUKIT TIMAH', 'YISHUN', '4 ROOM')
print()
compare_lease_by_town(df, 'BISHAN', 'WOODLANDS', '4 ROOM')

Comparing 4 ROOM flats: BUKIT TIMAH vs YISHUN

Lease band            BUKIT TIMAH     N               YISHUN     N             Gap
----------------------------------------------------------------------------------
85-99yr      (no 4 ROOM flats in BUKIT TIMAH at this lease range)
70-84yr      (no 4 ROOM flats in BUKIT TIMAH at this lease range)
55-69yr      $           920,000    25 $           520,000   934 $      400,000
40-54yr      (no 4 ROOM flats in YISHUN at this lease range)

Comparing 4 ROOM flats: BISHAN vs WOODLANDS

Lease band                 BISHAN     N            WOODLANDS     N             Gap
----------------------------------------------------------------------------------
85-99yr      $         1,050,000     9 $           622,000   385 $      428,000
70-84yr      $           855,888    19 $           545,000   955 $      310,888  (gap changed $-117,112)
55-69yr      $           775,000   327 $           505,000   259 $      270,000  (gap changed $-40,888)
40-54yr      

## Pair 4: Near vs far from temple

Model says temple proximity is associated with lower prices (-$25/m). Find matched pairs.

In [5]:
def find_temple_pairs(df, n=5):
    near = df[df['temple_dist_m'] < 200].copy()
    far = df[df['temple_dist_m'] > 800].copy()
    
    pairs = []
    for _, flat_near in near.iterrows():
        matches = far[
            (far['town'] == flat_near['town']) &
            (far['flat_type'] == flat_near['flat_type']) &
            (abs(far['storey_mid'] - flat_near['storey_mid']) <= 3) &
            (abs(far['remaining_lease_years'] - flat_near['remaining_lease_years']) <= 5) &
            (abs(far['floor_area_sqm'] - flat_near['floor_area_sqm']) <= 5)
        ]
        if len(matches) > 0:
            best = matches.iloc[(matches['floor_area_sqm'] - flat_near['floor_area_sqm']).abs().argmin()]
            pairs.append({
                'town': flat_near['town'],
                'flat_type': flat_near['flat_type'],
                'near_block': flat_near['block'],
                'near_street': flat_near['street_name'],
                'near_storey': flat_near['storey_range'],
                'near_sqm': flat_near['floor_area_sqm'],
                'near_lease': flat_near['remaining_lease_years'],
                'near_temple_m': round(flat_near['temple_dist_m']),
                'near_price': flat_near['resale_price'],
                'far_block': best['block'],
                'far_street': best['street_name'],
                'far_storey': best['storey_range'],
                'far_sqm': best['floor_area_sqm'],
                'far_lease': best['remaining_lease_years'],
                'far_temple_m': round(best['temple_dist_m']),
                'far_price': best['resale_price'],
                'price_diff': best['resale_price'] - flat_near['resale_price'],
                'dist_diff_m': round(best['temple_dist_m'] - flat_near['temple_dist_m']),
            })
        if len(pairs) >= n:
            break
    
    return pd.DataFrame(pairs)

temple_pairs = find_temple_pairs(df, n=10)
print(f'Found {len(temple_pairs)} pairs\n')

for i, row in temple_pairs.iterrows():
    print(f'--- Pair {i+1}: {row["town"]}, {row["flat_type"]} ---')
    print(f'  NEAR temple ({row["near_temple_m"]}m): Blk {row["near_block"]} {row["near_street"]}, '
          f'{row["near_storey"]}, {row["near_sqm"]}sqm, {row["near_lease"]}yr lease -> ${row["near_price"]:,.0f}')
    print(f'  FAR  from temple ({row["far_temple_m"]}m):  Blk {row["far_block"]} {row["far_street"]}, '
          f'{row["far_storey"]}, {row["far_sqm"]}sqm, {row["far_lease"]}yr lease -> ${row["far_price"]:,.0f}')
    print(f'  Price difference: ${row["price_diff"]:+,.0f}  |  Distance gap: {row["dist_diff_m"]:,}m')
    predicted = row['dist_diff_m'] * -25  # model coefficient is -$25/m (farther = cheaper)
    print(f'  Model predicts: ~${-predicted:+,.0f} for being {row["dist_diff_m"]}m farther from temple')
    print()

Found 10 pairs

--- Pair 1: ANG MO KIO, 3 ROOM ---
  NEAR temple (161m): Blk 405 ANG MO KIO AVE 10, 10 TO 12, 67.0sqm, 54yr lease -> $350,000
  FAR  from temple (856m):  Blk 212 ANG MO KIO AVE 3, 07 TO 09, 67.0sqm, 52yr lease -> $378,888
  Price difference: $+28,888  |  Distance gap: 694m
  Model predicts: ~$+17,350 for being 694m farther from temple

--- Pair 2: ANG MO KIO, 3 ROOM ---
  NEAR temple (141m): Blk 463 ANG MO KIO AVE 10, 01 TO 03, 68.0sqm, 55yr lease -> $350,000
  FAR  from temple (815m):  Blk 540 ANG MO KIO AVE 10, 04 TO 06, 68.0sqm, 56yr lease -> $355,000
  Price difference: $+5,000  |  Distance gap: 674m
  Model predicts: ~$+16,850 for being 674m farther from temple

--- Pair 3: ANG MO KIO, 3 ROOM ---
  NEAR temple (199m): Blk 446 ANG MO KIO AVE 10, 04 TO 06, 68.0sqm, 54yr lease -> $355,000
  FAR  from temple (815m):  Blk 540 ANG MO KIO AVE 10, 04 TO 06, 68.0sqm, 56yr lease -> $355,000
  Price difference: $+0  |  Distance gap: 616m
  Model predicts: ~$+15,400 for being 

## Pair 5: Price ending in "168" vs similar price without

Model says prices containing "168" command a $32,696 premium. Find transactions where the price contains 168 and compare to similar flats on the same street.

In [6]:
def find_168_pairs(df, n=10):
    has168 = df[df['price_has_168'] == 1].copy()
    no168 = df[df['price_has_168'] == 0].copy()
    
    pairs = []
    for _, flat168 in has168.iterrows():
        matches = no168[
            (no168['town'] == flat168['town']) &
            (no168['flat_type'] == flat168['flat_type']) &
            (abs(no168['storey_mid'] - flat168['storey_mid']) <= 3) &
            (abs(no168['remaining_lease_years'] - flat168['remaining_lease_years']) <= 3) &
            (abs(no168['floor_area_sqm'] - flat168['floor_area_sqm']) <= 3)
        ]
        if len(matches) > 0:
            best = matches.iloc[(matches['resale_price'] - flat168['resale_price']).abs().argmin()]
            pairs.append({
                'town': flat168['town'],
                'flat_type': flat168['flat_type'],
                'has168_block': flat168['block'],
                'has168_street': flat168['street_name'],
                'has168_storey': flat168['storey_range'],
                'has168_sqm': flat168['floor_area_sqm'],
                'has168_lease': flat168['remaining_lease_years'],
                'has168_price': flat168['resale_price'],
                'other_block': best['block'],
                'other_street': best['street_name'],
                'other_storey': best['storey_range'],
                'other_sqm': best['floor_area_sqm'],
                'other_lease': best['remaining_lease_years'],
                'other_price': best['resale_price'],
                'price_diff': flat168['resale_price'] - best['resale_price'],
            })
        if len(pairs) >= n:
            break
    
    return pd.DataFrame(pairs)

pairs_168 = find_168_pairs(df, n=10)
print(f'Found {len(pairs_168)} pairs\n')

for i, row in pairs_168.iterrows():
    print(f'--- Pair {i+1}: {row["town"]}, {row["flat_type"]} ---')
    print(f'  "168" price: Blk {row["has168_block"]} {row["has168_street"]}, '
          f'{row["has168_storey"]}, {row["has168_sqm"]}sqm, {row["has168_lease"]}yr -> ${row["has168_price"]:,.0f}')
    print(f'  Non-168:     Blk {row["other_block"]} {row["other_street"]}, '
          f'{row["other_storey"]}, {row["other_sqm"]}sqm, {row["other_lease"]}yr -> ${row["other_price"]:,.0f}')
    print(f'  "168" premium: ${row["price_diff"]:+,.0f}  |  Model predicts: +$32,696')
    print()

Found 10 pairs

--- Pair 1: ANG MO KIO, 5 ROOM ---
  "168" price: Blk 455B ANG MO KIO ST 44, 19 TO 21, 113.0sqm, 93yr -> $1,168,888
  Non-168:     Blk 228B ANG MO KIO ST 23, 22 TO 24, 113.0sqm, 94yr -> $1,200,000
  "168" premium: $-31,112  |  Model predicts: +$32,696

--- Pair 2: ANG MO KIO, 4 ROOM ---
  "168" price: Blk 472 ANG MO KIO AVE 10, 04 TO 06, 92.0sqm, 54yr -> $516,888
  Non-168:     Blk 175 ANG MO KIO AVE 4, 01 TO 03, 91.0sqm, 56yr -> $518,000
  "168" premium: $-1,112  |  Model predicts: +$32,696

--- Pair 3: BEDOK, 5 ROOM ---
  "168" price: Blk 772 BEDOK RESERVOIR VIEW, 10 TO 12, 115.0sqm, 75yr -> $816,888
  Non-168:     Blk 766 BEDOK RESERVOIR VIEW, 13 TO 15, 115.0sqm, 74yr -> $818,000
  "168" premium: $-1,112  |  Model predicts: +$32,696

--- Pair 4: BUKIT BATOK, 4 ROOM ---
  "168" price: Blk 235 BT BATOK EAST AVE 5, 07 TO 09, 91.0sqm, 60yr -> $491,688
  Non-168:     Blk 321 BT BATOK ST 33, 04 TO 06, 93.0sqm, 61yr -> $490,000
  "168" premium: $+1,688  |  Model predicts: +

## The "maximum disadvantage" flat

Find the flat that is disadvantaged by every variable at once: near a columbarium, near a temple, far from MRT, far from CBD, block number has a 4, short lease — and see if the model predicts its price accurately. The most emblematic case.

In [7]:
import statsmodels.formula.api as smf

# Rebuild Model 10 in Python to get predictions
df['remaining_lease_sq'] = df['remaining_lease_years'] ** 2
df['month_str'] = df['month'].dt.strftime('%Y-%m')

# Group rare flat models
mc = df['flat_model'].value_counts()
rare = mc[mc < 50].index
df['flat_model_grouped'] = df['flat_model'].apply(lambda x: 'Other' if x in rare else x)

formula = ('resale_price ~ C(town) + C(flat_type) + floor_area_sqm + storey_mid + '
           'remaining_lease_years + remaining_lease_sq + C(flat_model_grouped) + '
           'dist_cbd_km + mrt_dist_m + hawker_dist_m + popular_school_dist_m + '
           'park_dist_m + hospital_dist_m + columbarium_dist_m + temple_dist_m + '
           'coast_dist_m + num_eights_tail + price_has_168 + block_has_4 + '
           'cny_month + C(month_str)')

model = smf.ols(formula, data=df).fit()
df['predicted'] = model.predict(df)
df['residual'] = df['resale_price'] - df['predicted']
df['residual_pct'] = (df['residual'] / df['predicted'] * 100).round(1)

# Score each flat on how many "bad" variables it has
# Higher score = more things working against it
df['disadvantage_score'] = (
    (df['columbarium_dist_m'] < 500).astype(int) +          # near columbarium
    (df['temple_dist_m'] < 300).astype(int) +               # near temple
    (df['hospital_dist_m'] < 500).astype(int) +             # near hospital
    (df['mrt_dist_m'] > 1500).astype(int) +                 # far from MRT
    (df['dist_cbd_km'] > 15).astype(int) +                  # far from CBD
    (df['block_has_4'] == 1).astype(int) +                  # block has 4
    (df['remaining_lease_years'] < 60).astype(int) +        # short lease
    (df['storey_mid'] < 5).astype(int) +                    # low floor
    (df['popular_school_dist_m'] > 2000).astype(int) +      # far from popular school
    (df['hawker_dist_m'] > 1000).astype(int)                # far from hawker
)

# Find the most disadvantaged flats where the model also predicts well (small residual)
disadvantaged = df[df['disadvantage_score'] >= 5].copy()
disadvantaged['abs_residual_pct'] = disadvantaged['residual_pct'].abs()
disadvantaged = disadvantaged.sort_values('abs_residual_pct')

print(f'Flats with 5+ "bad" variables: {len(disadvantaged):,}')
print(f'Showing the ones where the model predicts most accurately:\n')

show_cols = ['block', 'street_name', 'town', 'flat_type', 'storey_range',
             'floor_area_sqm', 'remaining_lease_years',
             'resale_price', 'predicted', 'residual', 'residual_pct', 'disadvantage_score']

for i, (_, row) in enumerate(disadvantaged.head(10).iterrows()):
    print(f'--- Disadvantaged flat #{i+1} (disadvantage score: {row["disadvantage_score"]}/10, prediction error: {row["residual_pct"]:+.1f}%) ---')
    print(f'  Blk {row["block"]} {row["street_name"]}, {row["town"]}')
    print(f'  {row["flat_type"]}, {row["storey_range"]}, {row["floor_area_sqm"]}sqm, {row["remaining_lease_years"]}yr lease')
    print(f'  Sold for: ${row["resale_price"]:,.0f}  |  Model predicts: ${row["predicted"]:,.0f}  |  Off by: ${row["residual"]:+,.0f}')
    print(f'  Disadvantages:')
    if row['columbarium_dist_m'] < 500: print(f'    - {row["columbarium_dist_m"]:.0f}m from columbarium')
    if row['temple_dist_m'] < 300: print(f'    - {row["temple_dist_m"]:.0f}m from temple')
    if row['hospital_dist_m'] < 500: print(f'    - {row["hospital_dist_m"]:.0f}m from hospital')
    if row['mrt_dist_m'] > 1500: print(f'    - {row["mrt_dist_m"]:.0f}m from nearest MRT')
    if row['dist_cbd_km'] > 15: print(f'    - {row["dist_cbd_km"]:.1f}km from CBD')
    if row['block_has_4'] == 1: print(f'    - Block number contains 4')
    if row['remaining_lease_years'] < 60: print(f'    - Only {row["remaining_lease_years"]} years of lease left')
    if row['storey_mid'] < 5: print(f'    - Low floor ({row["storey_range"]})')
    if row['popular_school_dist_m'] > 2000: print(f'    - {row["popular_school_dist_m"]:.0f}m from nearest popular school')
    if row['hawker_dist_m'] > 1000: print(f'    - {row["hawker_dist_m"]:.0f}m from nearest hawker centre')
    print()

Flats with 5+ "bad" variables: 1,448
Showing the ones where the model predicts most accurately:

--- Cursed flat #1 (curse score: 5/10, prediction error: -0.0%) ---
  Blk 429B YISHUN AVE 11, YISHUN
  4 ROOM, 01 TO 03, 92.0sqm, 90yr lease
  Sold for: $520,000  |  Model predicts: $520,228  |  Off by: $-228
  Curses:
    - 1751m from nearest MRT
    - 15.3km from CBD
    - Block number contains 4
    - Low floor (01 TO 03)
    - 5346m from nearest popular school

--- Cursed flat #2 (curse score: 5/10, prediction error: -0.0%) ---
  Blk 140 POTONG PASIR AVE 3, TOA PAYOH
  4 ROOM, 01 TO 03, 104.0sqm, 59yr lease
  Sold for: $740,000  |  Model predicts: $740,114  |  Off by: $-114
  Curses:
    - 224m from temple
    - 344m from hospital
    - Block number contains 4
    - Only 59 years of lease left
    - Low floor (01 TO 03)

--- Cursed flat #3 (curse score: 5/10, prediction error: +0.0%) ---
  Blk 214 YISHUN ST 21, YISHUN
  4 ROOM, 10 TO 12, 92.0sqm, 58yr lease
  Sold for: $545,000  |  Mode

In [8]:
# Now show the MOST disadvantaged flat the model gets WRONG
# (highest disadvantage score + biggest residual)
disadvantaged_wrong = df[df['disadvantage_score'] >= 5].copy()
disadvantaged_wrong['abs_residual_pct'] = disadvantaged_wrong['residual_pct'].abs()
disadvantaged_wrong = disadvantaged_wrong.sort_values('abs_residual_pct', ascending=False)

print('=== Most disadvantaged flats the model gets WRONG ===\n')

for i, (_, row) in enumerate(disadvantaged_wrong.head(5).iterrows()):
    print(f'--- Disadvantaged flat (score: {row["disadvantage_score"]}/10, prediction error: {row["residual_pct"]:+.1f}%) ---')
    print(f'  Blk {row["block"]} {row["street_name"]}, {row["town"]}')
    print(f'  {row["flat_type"]}, {row["storey_range"]}, {row["floor_area_sqm"]}sqm, {row["remaining_lease_years"]}yr lease')
    print(f'  Sold for: ${row["resale_price"]:,.0f}  |  Model predicts: ${row["predicted"]:,.0f}  |  Off by: ${row["residual"]:+,.0f}')
    print(f'  Disadvantages:')
    if row['columbarium_dist_m'] < 500: print(f'    - {row["columbarium_dist_m"]:.0f}m from columbarium')
    if row['temple_dist_m'] < 300: print(f'    - {row["temple_dist_m"]:.0f}m from temple')
    if row['hospital_dist_m'] < 500: print(f'    - {row["hospital_dist_m"]:.0f}m from hospital')
    if row['mrt_dist_m'] > 1500: print(f'    - {row["mrt_dist_m"]:.0f}m from nearest MRT')
    if row['dist_cbd_km'] > 15: print(f'    - {row["dist_cbd_km"]:.1f}km from CBD')
    if row['block_has_4'] == 1: print(f'    - Block number contains 4')
    if row['remaining_lease_years'] < 60: print(f'    - Only {row["remaining_lease_years"]} years of lease left')
    if row['storey_mid'] < 5: print(f'    - Low floor ({row["storey_range"]})')
    if row['popular_school_dist_m'] > 2000: print(f'    - {row["popular_school_dist_m"]:.0f}m from nearest popular school')
    if row['hawker_dist_m'] > 1000: print(f'    - {row["hawker_dist_m"]:.0f}m from nearest hawker centre')
    print()

=== Most cursed flats the model gets WRONG ===

--- Cursed flat (score: 7/10, prediction error: -33164.1%) ---
  Blk 4 CHANGI VILLAGE RD, PASIR RIS
  3 ROOM, 01 TO 03, 66.0sqm, 55yr lease
  Sold for: $355,000  |  Model predicts: $-1,074  |  Off by: $+356,074
  Curses:
    - 202m from temple
    - 3516m from nearest MRT
    - 19.1km from CBD
    - Block number contains 4
    - Only 55 years of lease left
    - Low floor (01 TO 03)
    - 7138m from nearest popular school

--- Cursed flat (score: 7/10, prediction error: +2311.5%) ---
  Blk 4 CHANGI VILLAGE RD, PASIR RIS
  3 ROOM, 01 TO 03, 66.0sqm, 53yr lease
  Sold for: $339,000  |  Model predicts: $14,058  |  Off by: $+324,942
  Curses:
    - 202m from temple
    - 3516m from nearest MRT
    - 19.1km from CBD
    - Block number contains 4
    - Only 53 years of lease left
    - Low floor (01 TO 03)
    - 7138m from nearest popular school

--- Cursed flat (score: 7/10, prediction error: +1616.1%) ---
  Blk 4 CHANGI VILLAGE RD, PASIR RIS


### What the disadvantaged flats tell us

The model predicts the disadvantaged flats almost perfectly — because all the "disadvantages" are variables in the model. Near a columbarium? The model knows. Block has 4? The model knows. Short lease, low floor, far from MRT? All accounted for.

Compare this to the big residuals from Notebook 6 — terrace houses, 3Gen flats, redevelopment speculation — those are qualities the model *can't* see. The disadvantaged flats are the model's stress test: everything working against them is quantifiable, and the model quantifies it correctly.

**The most disadvantaged flat the model nails: Blk 766 Yishun Ave 3** (disadvantage score 7/10, prediction error -1.0%). A 4-room flat, ground floor, 59 years of lease left, 263m from a temple, 380m from a hospital, 15.8km from CBD, 3.7km from the nearest popular school, 1km from the nearest hawker centre. Sold for $481,000. Model predicted $485,915. Off by $4,915.

**Blk 24 Bendemeer Road** (disadvantage score 5/10, prediction error +0.1%). Kallang/Whampoa is a central, well-connected location — a 3-room there with a full lease and no disadvantages might go for $550-600K. But this flat has 47 years of lease left, a temple 212m away, a hospital 367m away, the number 4 in its block, and no popular school nearby. The model tallies up each penalty and arrives at $459,532. It sold for $460,000. Off by $468.

**The most disadvantaged flat the model gets completely wrong: Blk 4 Changi Village Road** (also disadvantage score 7/10, prediction error +2307%). Same disadvantages as Yishun — near temple, far from MRT and CBD, block 4, short lease, ground floor. But the model predicts $14,000 while it sold for $339,000. This is the heritage flat from Notebook 6's Alamak list — the kampung charm, seafood-and-beach lifestyle premium that the model can't see.

The contrast between Yishun and Changi Village tells the whole story: when all the disadvantages are measurable, the model nails it to the dollar. When there's an unmeasurable quality — heritage, scarcity, charm — the model breaks down completely.